# PingPro — Fine-tuning YOLOv8 : détection de balle de tennis de table

À exécuter sur **Google Colab (GPU gratuit : Runtime > Modifier le type d'exécution > T4 GPU)**.

1. Mettez votre clé API Roboflow (compte gratuit sur roboflow.com) dans `ROBOFLOW_API_KEY`.
2. Exécutez les cellules dans l'ordre.
3. Récupérez `best.pt` (sortie de la dernière cellule), puis en local :
   `python backend/tools/export_ball_model.py --weights best.pt`

Dataset par défaut : `madianou-kqrfk/table-tennis-ball-detection` (Roboflow Universe).
Vous pouvez le remplacer par tout dataset YOLO de balle de ping (cherchez "table tennis ball" sur Roboflow Universe).

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
from roboflow import Roboflow
ROBOFLOW_API_KEY = "COLLEZ_VOTRE_CLE_ICI"

WORKSPACE_PROJECT = "madianou-kqrfk/table-tennis-ball-detection"
VERSION = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
ds = rf.workspace(WORKSPACE_PROJECT.split("/")[0]).project(WORKSPACE_PROJECT.split("/")[1]).version(VERSION).download("yolov8")
DATA_YAML = f"{ds.location}/data.yaml"
print("Dataset :", DATA_YAML)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=DATA_YAML,
    epochs=60,
    imgsz=640,
    patience=15,
    degrees=5,
    scale=0.3,
    translate=0.1,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=10,
)

In [ ]:
# Métriques finales (mAP50) — attendez-vous à mAP50 > 0.85 sur un bon dataset
metrics = model.val()
print(metrics.box)

In [ ]:
# Export ONNX + téléchargement de best.pt
from google.colab import files
model.export(format="onnx", imgsz=640)
files.download(str(model.trainer.best))  # best.pt à exporter en local